# ECG DISEASE CLASSIFICATION
## 1D CNN + LSTM - Google Colab Version

### Install Dependencies

In [1]:
import subprocess
import sys

print("Installing required packages...\n")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow", "scikit-learn", "matplotlib", "numpy"])

print("Packages installed successfully!\n")

Installing required packages...

Packages installed successfully!



### Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

### Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!\n")
    COLAB = True
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.\n")
    COLAB = False

Not running in Google Colab. Skipping Drive mount.



### Dataset Configuration

In [ ]:
# Dataset path - UPDATE THIS TO YOUR DRIVE PATH
if COLAB:
   DATASET_PATH = "/content/drive/MyDrive/Final Year Project/datasets/mendely ecg version 2 .npz files"
else:
    DATASET_PATH = "/content/drive/MyDrive/Final Year Project/datasets/mendely ecg version 2 .npz files"

# Classes
classes = ["MI", "HB", "PMI", "Normal"]

# Parameters
MAX_LEN = 5000

### Load Dataset

In [ ]:
X = []
y = []

print("\nLoading dataset...\n")

for label in classes:
    folder = os.path.join(DATASET_PATH, label)

    if not os.path.exists(folder):
        print(f"Warning: Folder not found: {folder}")
        continue

    print("Reading:", folder)

    for file in os.listdir(folder):
        if file.endswith(".npz"):
            path = os.path.join(folder, file)

            try:
                data = np.load(path)

                # Extract 12 leads
                signal = np.stack([
                    data['I'],
                    data['II'],
                    data['III'],
                    data['aVR'],
                    data['aVL'],
                    data['aVF'],
                    data['V1'],
                    data['V2'],
                    data['V3'],
                    data['V4'],
                    data['V5'],
                    data['V6']
                ], axis=1)

                # Remove NaN
                signal = np.nan_to_num(signal)

                # Fix length
                if len(signal) > MAX_LEN:
                    signal = signal[:MAX_LEN]
                elif len(signal) < MAX_LEN:
                    pad = MAX_LEN - len(signal)
                    signal = np.pad(signal, ((0, pad), (0, 0)), mode='constant')

                X.append(signal)
                y.append(label)

            except Exception as e:
                print("Error in file:", path)
                print(e)

if len(X) == 0:
    print("\n⚠️  No data loaded! Please check your dataset path.")
    print(f"Looking for: {DATASET_PATH}")
else:
    X = np.array(X)
    y = np.array(y)

    print("\nDataset Loaded Successfully!")
    print("X shape:", X.shape)
    print("y shape:", y.shape)


Loading dataset...


⚠️  No data loaded! Please check your dataset path.
Looking for: /content/drive/MyDrive/Final Year Project/datasets/mendely ecg version 2 .npz files


### Encode Labels and Split Data

In [ ]:
# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_cat = to_categorical(y_encoded)

print("\nClasses:")
print(encoder.classes_)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

ValueError: zero-size array to reduction operation maximum which has no identity

### Build Model

In [ ]:
print("\nBuilding Model...\n")

model = Sequential()

# CNN BLOCK 1
model.add(
    Conv1D(
        filters=32,
        kernel_size=5,
        activation='relu',
        input_shape=(MAX_LEN, 12)
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# CNN BLOCK 2
model.add(
    Conv1D(
        filters=64,
        kernel_size=5,
        activation='relu'
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# LSTM
model.add(
    LSTM(
        64,
        return_sequences=False
    )
)

# Dropout
model.add(Dropout(0.3))

# Dense Layer
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

# Output Layer
model.add(
    Dense(
        len(classes),
        activation='softmax'
    )
)

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Train Model

In [ ]:
print("\nTraining Started...\n")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8,
    verbose=1
)

### Save Model

In [ ]:
if COLAB:
    model_path = "/content/drive/MyDrive/ecg_cnn_lstm_model.h5"
else:
    model_path = "ecg_cnn_lstm_model.h5"

model.save(model_path)

print(f"\nModel Saved Successfully at: {model_path}")

### Model Evaluation

In [ ]:
print("\nEvaluating Model...\n")

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='weighted')
recall = recall_score(y_true, y_pred_classes, average='weighted')
f1 = f1_score(y_true, y_pred_classes, average='weighted')

print("\n============================")
print("EVALUATION METRICS")
print("============================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

### Classification Report

In [ ]:
print("\nClassification Report\n")
print(
    classification_report(
        y_true,
        y_pred_classes,
        target_names=encoder.classes_
    )
)

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)

print("\nConfusion Matrix\n")
print(cm)

### Accuracy Graph

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Loss Graph

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Test Single File

In [ ]:
print("\nTesting Single NPZ File...\n")

# CHANGE FILE PATH HERE
if COLAB:
    test_file = "/content/drive/MyDrive/dataset/MI/sample1.npz"
else:
    test_file = "dataset/MI/sample1.npz"

try:
    data = np.load(test_file)

    signal = np.stack([
        data['I'],
        data['II'],
        data['III'],
        data['aVR'],
        data['aVL'],
        data['aVF'],
        data['V1'],
        data['V2'],
        data['V3'],
        data['V4'],
        data['V5'],
        data['V6']
    ], axis=1)

    signal = np.nan_to_num(signal)

    # Fix length
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    elif len(signal) < MAX_LEN:
        pad = MAX_LEN - len(signal)
        signal = np.pad(signal, ((0, pad), (0, 0)), mode='constant')

    # Add batch dimension
    signal = np.expand_dims(signal, axis=0)

    # Predict
    prediction = model.predict(signal)
    predicted_class = np.argmax(prediction)
    predicted_label = encoder.inverse_transform([predicted_class])

    # Results
    print("\nPrediction Probabilities:\n")
    print(prediction)

    print("\nPredicted Disease:")
    print(predicted_label[0])

    print("\nConfidence:")
    print(f"{np.max(prediction) * 100:.2f}%")

except FileNotFoundError:
    print(f"⚠️  Test file not found: {test_file}")
    print("Please update the test_file path to match your dataset location.")